# GEE Multi-Dataset Downloader & Downscaling Pipeline
Notebook ini didesain khusus untuk mengunduh dan menyelaraskan 3 dataset satelit/reanalisis utama dari Google Earth Engine (GEE):
1. **ERA5-Land Hourly** (`ECMWF/ERA5_LAND/HOURLY`) - Multi-variabel (Curah hujan, Suhu, Titik embun, Komponen Angin U/V, Tekanan Permukaan)
2. **GSMaP Operational v8** (`JAXA/GPM_L3/GSMaP/v8/operational`)
3. **GPM IMERG Final** (`NASA/GPM_L3/IMERG_V07`)

### Fitur Utama:
- **Dual Compatibility:** Dapat berjalan di environment **Lokal** maupun **Kaggle Notebook** secara otomatis.
- **GeoJSON Boundary:** Memotong data sesuai poligon batas wilayah (`33.05_kecamatan.geojson`).
- **Per-Variable & Multi-Band Monthly Stacking (`toBands()`):** Mempercepat unduhan hingga 30x lipat tanpa melampaui limit 1024 band GEE.
- **Struktur Hierarkis 4-Tier:** Menyiapkan folder `data/<jenis_data>/<tahun>/<tahun_bulan>/` secara sistematis.

In [ ]:
!pip install geemap earthengine-api rioxarray geopandas xarray netCDF4 requests --quiet

In [ ]:
import os
import json
import ee
import geemap
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
from shapely.validation import make_valid
from shapely.ops import transform, unary_union

# ==========================================
# 0. PARAMETER UTAMA PERIODE TAHUN & DIREKTORI
# ==========================================
tahun_mulai = 2000
tahun_selesai = 2026

BASE_DIR = os.getcwd()

# Deteksi Otomatis File GeoJSON (Lokal & Kaggle)
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        import glob
        found = glob.glob("/kaggle/input/**/*.geojson", recursive=True)
        file_geojson = found[0] if found else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

print(f"📌 Periode Unduhan Data : {tahun_mulai} s.d. {tahun_selesai}")
print(f"📌 Lokasi File GeoJSON   : {file_geojson}")
print(f"📌 Working Directory     : {BASE_DIR}")

# ==========================================
# 1. INISIALISASI GOOGLE EARTH ENGINE (GEE)
# ==========================================
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    from google.oauth2.service_account import Credentials
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Kaggle Service Account (GEE_KEY)")
except Exception as e:
    print(f"ℹ️ Inisialisasi Lokal / Standar GEE: {e}")
    try:
        ee.Initialize(project='staklimjerukagung')
        print("✅ Berhasil Inisialisasi GEE Lokal")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project='staklimjerukagung')
        print("✅ Berhasil Autentikasi & Inisialisasi GEE")

In [ ]:
# ==========================================
# 2. MEMBACA & MEMBERSIHKAN GEOMETRI GEOJSON
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    if geom is None or geom.is_empty: return None
    return geom.buffer(0)

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

# Ekstraksi Bounding Box (West, South, East, North)
bounds = gdf.total_bounds # [minx, miny, maxx, maxy]
buffer_deg = 0.01
ee_bbox = ee.Geometry.BBox(bounds[0] - buffer_deg, bounds[1] - buffer_deg, bounds[2] + buffer_deg, bounds[3] + buffer_deg)

geojson_fc = gdf.__geo_interface__
ee_boundary = ee.FeatureCollection(geojson_fc["features"])

print(f"✅ Batas Wilayah GeoJSON Berhasil Dimuat!")
print(f"   Extent Kebumen: Lon [{bounds[0]:.4f}, {bounds[2]:.4f}], Lat [{bounds[1]:.4f}, {bounds[3]:.4f}]")

In [ ]:
# ==========================================
# 3. DOWNLOADER FUNCTION: ERA5-LAND HOURLY (PER-VARIABLE STACKING)
# ==========================================
# Perbaikan Error: GEE membatasi max 1024 band per ekspor (4464 band jika 6 var sekaligus).
# Solusi: Mengunduh 6 variabel secara individual (744 band/var <= 1024) lalu digabung di NetCDF.

def unduh_era5_land_bulanan(tahun, bulan, output_base_dir):
    out_dir = os.path.join(output_base_dir, "era5_land", str(tahun), f"{tahun}_{bulan:02d}")
    os.makedirs(out_dir, exist_ok=True)
    nc_path = os.path.join(out_dir, f"era5_land_{tahun}_{bulan:02d}.nc")
    
    if os.path.exists(nc_path):
        print(f"[ERA5-Land {tahun}-{bulan:02d}] File sudah ada, dilewati...")
        return
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    tgl_akhir = f"{tahun+1}-01-01" if bulan == 12 else f"{tahun}-{bulan+1:02d}-01"
    
    print(f"[ERA5-Land {tahun}-{bulan:02d}] Mengunduh 6 variabel dari GEE...")
    try:
        col = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
               .filterBounds(ee_bbox)
               .filterDate(tgl_mulai, tgl_akhir))
        
        if col.size().getInfo() == 0:
            print(f"[ERA5-Land {tahun}-{bulan:02d}] Data tidak tersedia di GEE.")
            return
            
        timestamps = col.aggregate_array("system:time_start").getInfo()
        dates = pd.to_datetime(timestamps, unit='ms')
        
        # Konfigurasi Variabel & Konversi Unit
        vars_config = [
            ('precipitation', 'total_precipitation_hourly', lambda img: img.multiply(1000)),
            ('temperature_2m', 'temperature_2m', lambda img: img.subtract(273.15)),
            ('dewpoint_temperature_2m', 'dewpoint_temperature_2m', lambda img: img.subtract(273.15)),
            ('u_wind_10m', 'u_component_of_wind_10m', lambda img: img),
            ('v_wind_10m', 'v_component_of_wind_10m', lambda img: img),
            ('surface_pressure', 'surface_pressure', lambda img: img.divide(100))
        ]
        
        ds_out = xr.Dataset()
        temp_tifs = []
        
        for var_out_name, gee_band_name, transform_fn in vars_config:
            tif_var_temp = os.path.join(out_dir, f"temp_era5_{var_out_name}_{tahun}_{bulan:02d}.tif")
            temp_tifs.append(tif_var_temp)
            
            # Select single band & transform
            col_var = col.select(gee_band_name).map(lambda img: transform_fn(img).rename(var_out_name))
            stacked_var_img = col_var.toBands().clip(ee_bbox)
            
            # Export TIF per-variabel (max 744 band <= 1024)
            geemap.ee_export_image(
                stacked_var_img,
                filename=tif_var_temp,
                region=ee_bbox,
                scale=11132,
                file_per_band=False
            )
            
            # Load with rioxarray
            with rxr.open_rasterio(tif_var_temp, masked=True) as da_var:
                da_var = da_var.rename({'band': 'time'})
                da_var['time'] = dates[:len(da_var.time)]
                da_var.name = var_out_name
                ds_out[var_out_name] = da_var.load()
                
        # Simpan ke Multi-Variable NetCDF
        ds_out.to_netcdf(nc_path)
        ds_out.close()
        
        # Hapus file sementara
        for temp_tif in temp_tifs:
            if os.path.exists(temp_tif):
                os.remove(temp_tif)
                
        print(f"[ERA5-Land {tahun}-{bulan:02d}] ✓ Selesai tersimpan: {nc_path}")
    except Exception as e:
        print(f"[ERA5-Land {tahun}-{bulan:02d}] ❌ Error: {e}")
        for var_out_name, _, _ in vars_config:
            tif_var_temp = os.path.join(out_dir, f"temp_era5_{var_out_name}_{tahun}_{bulan:02d}.tif")
            if os.path.exists(tif_var_temp):
                os.remove(tif_var_temp)

In [ ]:
# ==========================================
# 4. DOWNLOADER FUNCTION: GSMaP OPERATIONAL V8
# ==========================================
def unduh_gsmap_bulanan(tahun, bulan, output_base_dir):
    out_dir = os.path.join(output_base_dir, "gsmap", str(tahun), f"{tahun}_{bulan:02d}")
    os.makedirs(out_dir, exist_ok=True)
    nc_path = os.path.join(out_dir, f"gsmap_{tahun}_{bulan:02d}.nc")
    tif_temp = os.path.join(out_dir, f"gsmap_{tahun}_{bulan:02d}_temp.tif")
    
    if os.path.exists(nc_path):
        print(f"[GSMaP {tahun}-{bulan:02d}] File sudah ada, dilewati...")
        return
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    tgl_akhir = f"{tahun+1}-01-01" if bulan == 12 else f"{tahun}-{bulan+1:02d}-01"
    
    print(f"[GSMaP {tahun}-{bulan:02d}] Mengunduh dari GEE...")
    try:
        col = (ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational")
               .filterBounds(ee_bbox)
               .filterDate(tgl_mulai, tgl_akhir)
               .select("hourlyPrecipRateGC"))
        
        if col.size().getInfo() == 0:
            print(f"[GSMaP {tahun}-{bulan:02d}] Data tidak tersedia di GEE.")
            return
            
        timestamps = col.aggregate_array("system:time_start").getInfo()
        dates = pd.to_datetime(timestamps, unit='ms')
        
        stacked_img = col.toBands().clip(ee_bbox)
        
        geemap.ee_export_image(
            stacked_img,
            filename=tif_temp,
            region=ee_bbox,
            scale=11132,
            file_per_band=False
        )
        
        with rxr.open_rasterio(tif_temp, masked=True) as da:
            da = da.rename({'band': 'time'})
            da['time'] = dates[:len(da.time)]
            da.name = "precipitation"
            da.to_netcdf(nc_path)
            
        if os.path.exists(tif_temp): os.remove(tif_temp)
        print(f"[GSMaP {tahun}-{bulan:02d}] ✓ Selesai tersimpan: {nc_path}")
    except Exception as e:
        print(f"[GSMaP {tahun}-{bulan:02d}] ❌ Error: {e}")
        if os.path.exists(tif_temp): os.remove(tif_temp)

In [ ]:
# ==========================================
# 5. DOWNLOADER FUNCTION: GPM IMERG FINAL (2-CHUNK HALF-MONTH STACKING)
# ==========================================
# Perbaikan Error: GPM IMERG 30-menit memuat ~1488 band per bulan (> limit 1024 GEE).
# Solusi: Mengunduh 2 chunk setengah bulan (H1: tgl 1-15 [720 band] & H2: tgl 16-end [768 band]).

def unduh_imerg_bulanan(tahun, bulan, output_base_dir):
    out_dir = os.path.join(output_base_dir, "imerg", str(tahun), f"{tahun}_{bulan:02d}")
    os.makedirs(out_dir, exist_ok=True)
    nc_path = os.path.join(out_dir, f"imerg_{tahun}_{bulan:02d}.nc")
    
    if os.path.exists(nc_path):
        print(f"[GPM IMERG {tahun}-{bulan:02d}] File sudah ada, dilewati...")
        return
        
    import calendar
    _, last_day = calendar.monthrange(tahun, bulan)
    
    print(f"[GPM IMERG {tahun}-{bulan:02d}] Mengunduh 2 chunk (setengah bulan) dari GEE...")
    try:
        chunks = [
            (f"{tahun}-{bulan:02d}-01", f"{tahun}-{bulan:02d}-16", "h1"),
            (f"{tahun}-{bulan:02d}-16", f"{tahun+1}-01-01" if bulan == 12 else f"{tahun}-{bulan+1:02d}-01", "h2")
        ]
        
        da_list = []
        temp_files = []
        
        for tgl_m, tgl_a, part_name in chunks:
            tif_temp = os.path.join(out_dir, f"temp_imerg_{tahun}_{bulan:02d}_{part_name}.tif")
            temp_files.append(tif_temp)
            
            col = (ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
                   .filterBounds(ee_bbox)
                   .filterDate(tgl_m, tgl_a)
                   .select("precipitation"))
            
            if col.size().getInfo() == 0:
                col = (ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
                       .filterBounds(ee_bbox)
                       .filterDate(tgl_m, tgl_a)
                       .select("precipitation"))
                       
            if col.size().getInfo() == 0:
                continue
                
            timestamps = col.aggregate_array("system:time_start").getInfo()
            dates = pd.to_datetime(timestamps, unit='ms')
            
            stacked_img = col.toBands().clip(ee_bbox)
            
            geemap.ee_export_image(
                stacked_img,
                filename=tif_temp,
                region=ee_bbox,
                scale=11132,
                file_per_band=False
            )
            
            with rxr.open_rasterio(tif_temp, masked=True) as da_part:
                da_part = da_part.rename({'band': 'time'})
                da_part['time'] = dates[:len(da_part.time)]
                da_list.append(da_part.load())
                
        if not da_list:
            print(f"[GPM IMERG {tahun}-{bulan:02d}] Data tidak tersedia di GEE.")
            return
            
        # Gabungkan H1 dan H2
        da_full = xr.concat(da_list, dim='time')
        da_full.name = "precipitation"
        da_full.to_netcdf(nc_path)
        
        # Hapus file temp
        for tf in temp_files:
            if os.path.exists(tf):
                os.remove(tf)
                
        print(f"[GPM IMERG {tahun}-{bulan:02d}] ✓ Selesai tersimpan: {nc_path}")
    except Exception as e:
        print(f"[GPM IMERG {tahun}-{bulan:02d}] ❌ Error: {e}")
        for tf in temp_files:
            if os.path.exists(tf):
                os.remove(tf)


In [ ]:
# ==========================================
# 6. EKSEKUSI PIPELINE DOWNSCALING MULTI-DATASET
# ==========================================
folder_data_base = os.path.join(BASE_DIR, "data")

print(f"\n{'='*70}")
print(f"MEMULAI UNDUHAN MULTI-DATASET GEE ({tahun_mulai} - {tahun_selesai})")
print(f"Produk: ERA5-Land, GSMaP, dan GPM IMERG")
print(f"Struktur Output: {folder_data_base}/<jenis_data>/<tahun>/<tahun_bulan>/")
print(f"{'='*70}\n")

for thn in range(tahun_mulai, tahun_selesai + 1):
    for bln in range(1, 13):
        print(f"\n--- Periode {thn}-{bln:02d} ---")
        
        # 1. ERA5-Land
        try:
            unduh_era5_land_bulanan(thn, bln, folder_data_base)
        except Exception as e:
            print(f"  ❌ Gagal ERA5-Land {thn}-{bln:02d}: {e}")
            
        # 2. GSMaP
        try:
            unduh_gsmap_bulanan(thn, bln, folder_data_base)
        except Exception as e:
            print(f"  ❌ Gagal GSMaP {thn}-{bln:02d}: {e}")
            
        # 3. GPM IMERG
        try:
            unduh_imerg_bulanan(thn, bln, folder_data_base)
        except Exception as e:
            print(f"  ❌ Gagal GPM IMERG {thn}-{bln:02d}: {e}")

print(f"\n{'='*70}")
print("🎉 SELURUH PROSES UNDUHAN MULTI-DATASET SELESAI!")
print(f"{'='*70}")
